In [1]:
%cd ..

/Users/po3na4skld/PycharmProjects/Personal/SynSteg


/Users/po3na4skld/PycharmProjects/Binerals/bin_datasets_reading/venv/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import openai
import random
import os

os.environ["OPENAI_API_KEY"] = "os.environ["OPENAI_API_KEY"]"

from steganography import helper
from steganography.secret_key import (
    binarize_synonyms, 
    align_container_and_secret_key, 
    fix_token_container_size,
    binarize_synonyms_partially,
)

from utils.constants import TOPICS
from steganography.gpt import get_openai_output, get_openai_json_output
from utils.prompts import CONTAINER_GENERATION_INPUT, ALL_SYNONYMS_GENERATION_INPUT
from steganography.core import decode_message

from steganography.gpt import generate_container

In [6]:
BITS_PER_WORD = min(5, 5)
ADDITIONAL_BITS = 0
SAFETY_BUFFER = BITS_PER_WORD

In [7]:
test_message = "This is only the beginning of this project!"
binary_message = helper.binarize_message(test_message)

In [8]:
if ADDITIONAL_BITS:
    chunk_size = BITS_PER_WORD + ADDITIONAL_BITS 
    binary_message_chunks = list(helper.divide_chunks(binary_message, chunk_size))

In [12]:
container_length = len(binary_message) // (BITS_PER_WORD + ADDITIONAL_BITS) + SAFETY_BUFFER
container, usage = generate_container(container_length)

In [13]:
container

'The role of government in promoting healthy lifestyles is crucial for the well-being of its citizens. By implementing policies that encourage exercise and nutritious eating, governments can reduce healthcare costs and enhance public health. Initiatives like building parks, supporting community sports programs, and regulating food labeling empower individuals to make informed choices. Education campaigns to raise awareness about the benefits of healthy living play a significant role as well. Governments also have a responsibility to ensure access to affordable healthcare services, allowing everyone to maintain a healthy lifestyle regardless of economic background.'

In [14]:
container_splits = list(helper.divide_chunks(container.split(), 5))  # container.split(".")
len(container_splits)

19

In [15]:
ALL_SYNONYMS_GENERATION_PROMPT = """
Act as a professional linguist with 30 years of experience in this field.
You will be given a context (list of words). 

Eg: ["A", "simple", "example" "of" "the", "input"]

Your current task is to create a list of synonyms or replacement words for each word so that:
 - For each word, there should be strictly N_SYNONYMS synonyms or replacement words.
 - Ensure that synonyms or replacement words perfectly match the context (previous and next words).
 - Maintain the capitalization of synonyms or replacement words like in original word.
 - Created list of synonyms or replacement words for specific word should not contain that specific word.
 - No need to create synonyms or replacement words for numbers.

Always return your response in JSON format:
```json
{
    "words": [
        {"word_1": ["synonym_1", "synonym_2", "synonym_3", ..., "synonym_16"]}, 
        {"word_2": ["synonym_1", "synonym_2", "synonym_3", "synonym_4"]},
        ...,
        {"word_n": ["synonym_1", "synonym_2"]}
    ]
}
```
"""

In [16]:
synonyms_splits =[]
for container_split in container_splits:
    input_message = ALL_SYNONYMS_GENERATION_INPUT.format(context=container_split)
    synonyms = get_openai_json_output(ALL_SYNONYMS_GENERATION_PROMPT.replace("N_SYNONYMS", str(2**BITS_PER_WORD)), input_message, "words", 0.7)
    synonyms_splits += synonyms

JSONDecodeError: Expecting value: line 3110 column 1 (char 4374)

In [ ]:
synonyms_map, is_filled = [], False
for idx, synonym in enumerate(synonyms_splits):
    if is_filled:
        break
        
    for key, value in synonym.items():  # Always only one cycle
        if ADDITIONAL_BITS:
            try:
                partially_binarized = binarize_synonyms_partially(list(set(value)), binary_message_chunks[idx])
                synonyms_map.append({key: partially_binarized})
            except IndexError:
                is_filled = True
        else:
            synonyms_map.append({key: binarize_synonyms(key, value)})

In [ ]:
aligned_synonyms_map = align_container_and_secret_key(container, synonyms_map)

In [ ]:
encoded_message, current_idx = [], 0

for token, replacement_token in zip(container.split(), aligned_synonyms_map):
    ends_with_special, token = helper.check_endswith_special(token)
    if helper.is_token_replacable(token, replacement_token) and current_idx < len(binary_message):  # Ingest token
        if helper.has_duplicates(replacement_token):
            encoded_message.append(token + ends_with_special)
            continue

        token, was_capital = helper.check_capitalization(token, replacement_token)
        token_container_size = len(list(replacement_token[token].keys())[0])
        message_part = binary_message[current_idx:current_idx + token_container_size]
        if len(message_part) != token_container_size:
            replacement_token[token] = fix_token_container_size(replacement_token[token], len(message_part))
            
        new_token = replacement_token[token][message_part]
        if was_capital:
            new_token = new_token.capitalize()

        encoded_message.append(new_token + ends_with_special)
        current_idx += token_container_size
    else:  # Skipping token
        encoded_message.append(token + ends_with_special)

In [ ]:
encoded_container = ' '.join(encoded_message)

In [ ]:
decoded_message = decode_message(encoded_container, aligned_synonyms_map)

In [ ]:
decoded_message